In [1]:
import uproot
import pandas as pd

# Define the file path
file_path = "clusters_seeds_island_79507-0.root_ntuplizer.root"

print("Loading ML Input Features from ROOT file...")
with uproot.open(file_path) as root_file:
    # We only extract the exact branches needed for AI Training + Metadata
    branches_to_import = [
        "event", "hitID",      # Metadata (For batching and final scoring)
        "x", "y", "z",         # Features (3D Physical Space)
        "adc", "layer"         # Features (Energy and Topology)
    ]
    
    # Load the data directly into a Pandas DataFrame
    df_ml_input = root_file["ntp_hit"].arrays(branches_to_import, library="pd")

# Filter out dead pads (0 energy)
df_ml_input = df_ml_input[df_ml_input["adc"] > 0].reset_index(drop=True)

# -------------------------------------------------------------------------
# NOTE: At this point, you would run the KD-Tree Grid Join script from
# our previous step to generate the 'is_verified' label column!
# -------------------------------------------------------------------------

# Assuming you have generated 'is_verified':
# print("Exporting clean dataset for PyTorch DataLoader...")
# df_ml_input.to_csv("sPHENIX_ML_Training_Dataset.csv", index=False)

Loading ML Input Features from ROOT file...


In [2]:
import uproot
import awkward as ak
import numpy as np

def summarize_event_ecology(f, event_id):
    # 1. Load data for a specific event
    cut = f"event == {event_id}"

# ntp_hit: Raw hits grouped into 'islands' (hitID)
    hits = f["ntp_hit"].arrays(["hitID", "adc"], cut=cut)
# ntp_cluster: may be Centroids of islands (Filtered Clusters)
    clusters = f["ntp_cluster"].arrays(["adc"], cut=cut)    # any branches work with the same result
# ntp_clus_trk: Clusters successfully fitted to tracks
    clus_trk = f["ntp_clus_trk"].arrays(["seedID"], cut=cut)

    print(f"=== Ecology Summary for Event {event_id} ===")

 # Hit Level
# Note: hitID resets per event, and we filter adc > 0
    unique_islands = np.unique(hits.hitID[hits.adc > 0])
    print(f"ntp_hit:      {len(hits):>8} raw hit entries")
    print(f"              {len(unique_islands):>8} unique 'islands' (hitID with adc>0)")

# Cluster Level
    print(f"ntp_cluster:  {len(clusters):>8} total clusters (centroids)")
     
# Track Level
    unique_tracks = np.unique(clus_trk.seedID)
    print(f"ntp_clus_trk: {len(clus_trk):>8} clusters associated with tracks")
    print(f"              {len(unique_tracks):>8} unique reconstructed tracks (seedID)")


file_path = "clusters_seeds_island_79507-0.root_ntuplizer.root"
with uproot.open(file_path) as f:
    summarize_event_ecology(f, event_id=1)

=== Ecology Summary for Event 1 ===
ntp_hit:         72651 raw hit entries
                 23825 unique 'islands' (hitID with adc>0)
ntp_cluster:      9712 total clusters (centroids)
ntp_clus_trk:      837 clusters associated with tracks
                    26 unique reconstructed tracks (seedID)
